# 🚀 Cookbook 4: Production Patterns - Memory, Hooks & Deployment

This advanced cookbook covers patterns for production-ready agents!

**Topics covered:**
1. **Session Management** - Persisting conversations across restarts
2. **AgentCore Memory** - Advanced long-term memory with AWS
3. **Hooks** - Extending agent behavior (logging, validation, retries)
4. **Structured Output** - Type-safe responses with Pydantic
5. **AWS Lambda Deployment** - Deploying agents to production

**Prerequisites**: Cookbooks 1-3

---

In [ ]:
# Install packages
!pip install strands-agents strands-agents-tools pydantic -q
print("✅ Core packages ready!")

---
## Part 1: Session Management - Persisting Conversations
---

Session management lets you save and restore agent state across restarts. This is critical for:
- Multi-session conversations
- Stateful applications
- Crash recovery

### 💾 FileSessionManager - Local Storage

In [ ]:
from strands import Agent
from strands.session.file_session_manager import FileSessionManager
import os

# Create a session manager with a unique session ID
session_manager = FileSessionManager(
    session_id="user-alice-12345",
    storage_dir="./sessions"  # Where to store session files
)

# Create agent with session management
agent = Agent(
    session_manager=session_manager,
    system_prompt="You are a helpful assistant that remembers our conversations."
)

# Have a conversation
agent("Hi! My name is Alice and I love Python programming.")
agent("What's my favorite programming language?")

print(f"\n📁 Session stored at: ./sessions/")
print(f"✅ Messages will persist across restarts!")

### ☁️ S3SessionManager - Cloud Storage

For production, use S3 for distributed storage:

In [ ]:
# Uncomment to use S3 (requires AWS credentials)
# from strands import Agent
# from strands.session.s3_session_manager import S3SessionManager

# session_manager = S3SessionManager(
#     session_id="user-alice-12345",
#     bucket="my-agent-sessions",
#     prefix="production/",
#     region_name="us-west-2"
# )

# agent = Agent(
#     session_manager=session_manager,
#     system_prompt="You are a helpful assistant."
# )

print("S3 session manager example (requires AWS setup)")

---
## Part 2: AgentCore Memory - Advanced Long-Term Memory ⭐
---

**Amazon Bedrock AgentCore Memory** provides intelligent memory with:
- **Short-term memory (STM)**: Conversation persistence
- **Long-term memory (LTM)**: Learning user preferences, facts, and summaries
- **Semantic retrieval**: Find relevant memories based on meaning

In [ ]:
# Install AgentCore Memory package
!pip install 'bedrock-agentcore[strands-agents]' -q
print("✅ AgentCore Memory package installed!")

### 🧠 Setting Up AgentCore Memory

**Note**: This requires AWS setup and creating a memory resource first.

In [ ]:
# Example: Using AgentCore Memory (requires AWS setup)

# Step 1: Create a memory resource (one-time setup)
# from bedrock_agentcore.memory import MemoryClient
# 
# client = MemoryClient(region_name="us-east-1")
# memory = client.create_memory(
#     name="MyAgentMemory",
#     description="Memory for my production agent"
# )
# memory_id = memory.get('id')

# Step 2: Use with Strands Agent
# from datetime import datetime
# from strands import Agent
# from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
# from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

# config = AgentCoreMemoryConfig(
#     memory_id="your-memory-id",
#     session_id=f"session_{datetime.now().strftime('%Y%m%d%H%M%S')}",
#     actor_id="user-alice"
# )

# # Use context manager to ensure messages are flushed
# with AgentCoreMemorySessionManager(
#     agentcore_memory_config=config,
#     region_name="us-east-1"
# ) as session_manager:
#     agent = Agent(
#         system_prompt="You are a helpful assistant. Use what you know about the user.",
#         session_manager=session_manager
#     )
#     
#     agent("I love sushi with tuna")
#     agent("What should I have for lunch today?")

print("AgentCore Memory example (see code comments for implementation)")

### 🎯 Long-Term Memory Strategies

AgentCore Memory supports three built-in strategies:

| Strategy | Purpose | Namespace Pattern |
|----------|---------|-------------------|
| **summaryMemoryStrategy** | Summarizes conversation sessions | `/summaries/{actorId}/{sessionId}` |
| **userPreferenceMemoryStrategy** | Learns user preferences | `/preferences/{actorId}` |
| **semanticMemoryStrategy** | Extracts and stores facts | `/facts/{actorId}` |

In [ ]:
# Example: Creating memory with LTM strategies (one-time setup)

# from bedrock_agentcore.memory import MemoryClient
# 
# client = MemoryClient(region_name="us-east-1")
# comprehensive_memory = client.create_memory_and_wait(
#     name="ComprehensiveAgentMemory",
#     description="Full-featured memory with all strategies",
#     strategies=[
#         {
#             "summaryMemoryStrategy": {
#                 "name": "SessionSummarizer",
#                 "namespaces": ["/summaries/{actorId}/{sessionId}"]
#             }
#         },
#         {
#             "userPreferenceMemoryStrategy": {
#                 "name": "PreferenceLearner",
#                 "namespaces": ["/preferences/{actorId}"]
#             }
#         },
#         {
#             "semanticMemoryStrategy": {
#                 "name": "FactExtractor",
#                 "namespaces": ["/facts/{actorId}"]
#             }
#         }
#     ]
# )

print("LTM strategies example (see code comments)")

---
## Part 3: Hooks - Extending Agent Behavior ⭐
---

**Hooks** let you extend agent functionality by subscribing to lifecycle events. Use cases:
- Logging and monitoring
- Input/output validation
- Tool call interception
- Retry logic
- Metrics collection

### 📊 Basic Hook: Logging

In [ ]:
from strands import Agent
from strands.hooks import (
    HookProvider, 
    HookRegistry,
    BeforeInvocationEvent,
    AfterInvocationEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent
)

class LoggingHook(HookProvider):
    """A hook that logs all agent activity."""
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.log_request_start)
        registry.add_callback(AfterInvocationEvent, self.log_request_end)
        registry.add_callback(BeforeToolCallEvent, self.log_tool_start)
        registry.add_callback(AfterToolCallEvent, self.log_tool_end)
    
    def log_request_start(self, event: BeforeInvocationEvent) -> None:
        print(f"📥 Request started for agent: {event.agent.name or 'unnamed'}")
    
    def log_request_end(self, event: AfterInvocationEvent) -> None:
        print(f"📤 Request completed for agent: {event.agent.name or 'unnamed'}")
    
    def log_tool_start(self, event: BeforeToolCallEvent) -> None:
        print(f"  🔧 Tool starting: {event.tool_use['name']}")
    
    def log_tool_end(self, event: AfterToolCallEvent) -> None:
        status = event.result.get('status', 'unknown')
        print(f"  ✅ Tool completed: {event.tool_use['name']} ({status})")

# Create agent with logging hook
from strands_tools import calculator

agent = Agent(
    name="MathAgent",
    tools=[calculator],
    hooks=[LoggingHook()]
)

# Test it
print("\n--- Running agent with logging hook ---\n")
response = agent("What is 42 * 17?")

### 🛡️ Hook: Tool Rate Limiting

In [ ]:
from strands import Agent
from strands.hooks import HookProvider, HookRegistry, BeforeInvocationEvent, BeforeToolCallEvent
from threading import Lock

class LimitToolCounts(HookProvider):
    """Limits the number of times tools can be called per invocation."""
    
    def __init__(self, max_tool_counts: dict):
        """
        Args:
            max_tool_counts: Dict mapping tool names to max call counts.
        """
        self.max_tool_counts = max_tool_counts
        self.tool_counts = {}
        self._lock = Lock()
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset_counts)
        registry.add_callback(BeforeToolCallEvent, self.check_limit)
    
    def reset_counts(self, event: BeforeInvocationEvent) -> None:
        with self._lock:
            self.tool_counts = {}
    
    def check_limit(self, event: BeforeToolCallEvent) -> None:
        tool_name = event.tool_use["name"]
        
        with self._lock:
            max_count = self.max_tool_counts.get(tool_name)
            current_count = self.tool_counts.get(tool_name, 0) + 1
            self.tool_counts[tool_name] = current_count
        
        if max_count and current_count > max_count:
            # Cancel the tool call!
            event.cancel_tool = (
                f"Tool '{tool_name}' has been called too many times. "
                f"Limit is {max_count} per request."
            )
            print(f"⚠️ Tool '{tool_name}' rate limited!")

# Create agent with rate limiting
from strands_tools import calculator

agent = Agent(
    tools=[calculator],
    hooks=[LimitToolCounts({"calculator": 2})]  # Max 2 calculator calls
)

# Test - this will hit the limit
print("Testing rate limiting (max 2 calculator calls):")
response = agent("Calculate: 1+1, 2+2, 3+3, and 4+4")

### 🔒 Hook: Fixed Tool Arguments

Enforce security policies by overriding tool parameters:

In [ ]:
from strands.hooks import HookProvider, HookRegistry, BeforeToolCallEvent
from typing import Any

class ConstantToolArguments(HookProvider):
    """Force specific argument values for tools."""
    
    def __init__(self, fixed_arguments: dict):
        """
        Args:
            fixed_arguments: Dict mapping tool names to fixed parameter values.
        """
        self.fixed_arguments = fixed_arguments
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.fix_arguments)
    
    def fix_arguments(self, event: BeforeToolCallEvent) -> None:
        tool_name = event.tool_use["name"]
        
        if tool_name in self.fixed_arguments:
            # Override the tool input with fixed values
            tool_input = event.tool_use["input"]
            tool_input.update(self.fixed_arguments[tool_name])
            print(f"🔒 Fixed arguments for '{tool_name}': {self.fixed_arguments[tool_name]}")

# Example: Force calculator to always use precision=2
from strands import Agent
from strands_tools import calculator

agent = Agent(
    tools=[calculator],
    hooks=[ConstantToolArguments({"calculator": {"precision": 2}})]
)

print("Testing fixed arguments:")
response = agent("What is 22/7 with lots of decimal places?")

---
## Part 4: Structured Output with Pydantic ⭐
---

Get **type-safe, validated responses** from your agents using Pydantic models!

In [ ]:
from strands import Agent
from pydantic import BaseModel, Field

# Define your output structure
class PersonInfo(BaseModel):
    """Information about a person."""
    name: str = Field(description="Full name of the person")
    age: int = Field(description="Age in years", ge=0, le=150)
    occupation: str = Field(description="Current job or occupation")

# Create agent and request structured output
agent = Agent()

result = agent(
    "John Smith is a 30 year-old software engineer.",
    structured_output_model=PersonInfo
)

# Access the validated, typed object!
person: PersonInfo = result.structured_output

print(f"Name: {person.name}")
print(f"Age: {person.age}")
print(f"Occupation: {person.occupation}")
print(f"\nType: {type(person)}")

### 📋 More Complex Structured Output

In [ ]:
from strands import Agent
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum

class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class Task(BaseModel):
    """A task or todo item."""
    title: str = Field(description="Short title of the task")
    description: str = Field(description="Detailed description")
    priority: Priority = Field(description="Priority level")
    estimated_hours: float = Field(description="Estimated hours to complete", ge=0)
    tags: List[str] = Field(description="Tags for categorization", default=[])

class TaskList(BaseModel):
    """A list of tasks."""
    tasks: List[Task] = Field(description="List of tasks")
    total_estimated_hours: float = Field(description="Total estimated hours")

# Extract structured data from unstructured text
agent = Agent()

result = agent(
    """Create a task list from this:
    - Fix critical login bug (8 hours, security, backend)
    - Update documentation (2 hours, docs, low priority)
    - Implement new dashboard feature (16 hours, frontend, high priority)
    """,
    structured_output_model=TaskList
)

task_list: TaskList = result.structured_output

print(f"Total tasks: {len(task_list.tasks)}")
print(f"Total hours: {task_list.total_estimated_hours}")
print("\nTasks:")
for task in task_list.tasks:
    print(f"  - [{task.priority.value.upper()}] {task.title} ({task.estimated_hours}h)")

### 🔧 Structured Output + Tools

In [ ]:
from strands import Agent
from strands_tools import calculator
from pydantic import BaseModel, Field

class MathResult(BaseModel):
    """Result of a math operation."""
    operation: str = Field(description="The operation performed")
    operands: list = Field(description="The numbers involved")
    result: float = Field(description="The calculated result")
    explanation: str = Field(description="Brief explanation")

agent = Agent(tools=[calculator])

result = agent(
    "Calculate 15% tip on a $85.50 bill",
    structured_output_model=MathResult
)

math_result: MathResult = result.structured_output

print(f"Operation: {math_result.operation}")
print(f"Result: ${math_result.result:.2f}")
print(f"Explanation: {math_result.explanation}")

---
## Part 5: AWS Lambda Deployment
---

Here's how to deploy your agents to AWS Lambda for production use.

### 📦 Lambda Handler Pattern

In [ ]:
# Example Lambda handler (agent_handler.py)

lambda_handler_code = '''
from strands import Agent
from strands_tools import http_request
from typing import Dict, Any

# Define system prompt
SYSTEM_PROMPT = """You are a helpful assistant with HTTP capabilities.
You can make API requests to retrieve information."""

def handler(event: Dict[str, Any], context) -> str:
    """Lambda handler function."""
    
    # Create agent (created per-request for stateless operation)
    agent = Agent(
        system_prompt=SYSTEM_PROMPT,
        tools=[http_request],
    )
    
    # Get prompt from event
    prompt = event.get("prompt", "Hello!")
    
    # Run agent
    response = agent(prompt)
    
    return str(response)
'''

print("Lambda Handler Pattern:")
print("="*50)
print(lambda_handler_code)

### 🔧 Using the Official Lambda Layer

Strands provides official Lambda layers for quick deployment:

```
arn:aws:lambda:{region}:856699698935:layer:strands-agents-py{python_version}-{architecture}:{layer_version}
```

**Examples:**
- `arn:aws:lambda:us-east-1:856699698935:layer:strands-agents-py3_12-x86_64:1`
- `arn:aws:lambda:us-west-2:856699698935:layer:strands-agents-py3_12-aarch64:1`

**Supported:**
- Python: 3.10, 3.11, 3.12, 3.13
- Architectures: x86_64, aarch64
- Regions: us-east-1, us-west-2, eu-west-1, and more

### 📋 CDK Deployment Example

In [ ]:
cdk_example = '''
// TypeScript CDK Stack
import * as lambda from "aws-cdk-lib/aws-lambda";
import * as iam from "aws-cdk-lib/aws-iam";

// Define the Lambda function
const agentFunction = new lambda.Function(this, "AgentLambda", {
  runtime: lambda.Runtime.PYTHON_3_12,
  functionName: "MyAgentFunction",
  handler: "agent_handler.handler",
  code: lambda.Code.fromAsset("./lambda"),
  timeout: Duration.seconds(30),
  memorySize: 256,
  architecture: lambda.Architecture.ARM_64,
  
  // Use official Strands layer
  layers: [
    lambda.LayerVersion.fromLayerVersionArn(
      this, "StrandsLayer",
      "arn:aws:lambda:us-east-1:856699698935:layer:strands-agents-py3_12-aarch64:1"
    )
  ],
});

// Add Bedrock permissions
agentFunction.addToRolePolicy(
  new iam.PolicyStatement({
    actions: ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
    resources: ["*"],
  })
);
'''

print("CDK Deployment Example:")
print("="*50)
print(cdk_example)

---
## 📚 Summary

In this advanced cookbook, you learned:

1. **Session Management** - FileSessionManager and S3SessionManager for persistence
2. **AgentCore Memory** - Long-term memory with STM, LTM, and semantic retrieval
3. **Hooks** - Logging, rate limiting, and argument override patterns
4. **Structured Output** - Type-safe Pydantic responses from agents
5. **AWS Lambda** - Production deployment patterns and official layers

**Key Takeaways:**
- Use **hooks** to add monitoring, validation, and custom behavior
- Use **structured output** when you need reliable, typed responses
- Use **session managers** for stateful, persistent conversations
- Use **AgentCore Memory** for intelligent long-term memory

**Next**: Check out the official Strands documentation for more deployment patterns!